In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)
sys.path.insert(0, os.path.abspath('../../..'))

# UC2 Analyse: Altersbasierte digitale Kompetenz und Nutzungsverhalten

Dieses Notebook fasst die Ergebnisse der UC2-Szenarien zusammen:
- **Teil A - Ältere (65+)**: baseline, phone_support, simple_ui, full_support
- **Teil B - Junge Nutzer**: students_15_20, students_21_30

## 🔧 Simulationssetup und Kontext

Diese Studie basiert auf der **Klagenfurt Infrastruktur** und untersucht, wie digitale Kompetenz die Nutzbarkeit von On-Demand-Mobilität beeinflusst.

### Verknüpfung zu H1/H2

**Warum 20 Fahrzeuge?**
- **H1-Baseline:** 20 Fahrzeuge bieten **84% Servicerate** bei **100% Fahrzeugauslastung**
- **H2-Schwellenwert:** 20 Fahrzeuge liegen **über der kritischen Grenze** von 16 Fahrzeugen (<70% Service)
- **UC2-Design:** Untersucht **qualitative Verbesserungen** (Support-Maßnahmen) bei **konstanter Kapazität**
- **Vorteil:** Isoliert den Effekt der digitalen Kompetenz von reinen Kapazitätseffekten

### Netzwerk und Infrastruktur
- **Knoten:** 2.099 (Straßenkreuzungen und Haltestellen)
- **Kanten:** 5.658 (Straßenverbindungen)
- **Zonen:** 12 (K-Means-Clustering für optimale Alonso-Mora Performance)
- **Zentroiden:** 103 (strategische Ein-/Ausstiegspunkte)
- **Simulationszeitraum:** 7:00-9:00 Uhr (Morgenspitze)
- **Nachfrage:** 100 Mobilitätsanfragen

### Algorithmus-Konfiguration
- **Optimierung:** Alonso-Mora Batch Assignment (State-of-the-art Ridepooling)
- **Reoptimierung:** Alle 30 Sekunden
- **Max. Wartezeit:** 8 Minuten
- **Max. Umwegfaktor:** 40% der direkten Fahrzeit
- **Fahrzeugkapazität:** 8 Passagiere

### UC2-Spezifische Parameter

Die vier Szenarien unterscheiden sich in folgenden Parametern (abbilden digitale Kompetenz):

| Parameter | Bedeutung | Baseline | Telefon | Simple UI | Full Support |
|-----------|-----------|----------|---------|-----------|--------------|
| `user_max_decision_time` | Entscheidungsfenster | 60s | 240s | 120s | 300s |
| `user_max_wait_time` | Wartezeit-Toleranz (Grenze 1) | 180s | 180s | 240s | 300s |
| `user_max_wait_time_2` | Wartezeit-Toleranz (Grenze 2) | 300s | 300s | 420s | 480s |
| `walking_speed` | Gehgeschwindigkeit | 1.0 m/s | 1.0 m/s | 1.0 m/s | 1.0 m/s |
| `max_walking_dist` | Max. Gehweg | 200m | 200m | 200m | 250m |
| `op_const_boarding_time` | Boarding-Zeit | 60s | 50s | 50s | 30s |

**Interpretation:**
- **Baseline:** Niedrige digitale Kompetenz → kurzes Entscheidungsfenster, geringe Toleranz
- **Telefon-Support:** Mehr Zeit für Entscheidung (durch Hilfe am Telefon)
- **Simple UI:** Höhere Toleranz (besseres Verständnis durch vereinfachte App)
- **Full Support:** Kombination aller Maßnahmen + physische Assistenz (kürzere Boarding-Zeit)

## 1. Import der benötigten Bibliotheken
Wir verwenden pandas und plotly für die Auswertung und Visualisierung.

In [2]:
import pandas as pd
import plotly.express as px
import os
from glob import glob

## 2. Ergebnisse laden und Kennzahlen extrahieren
Wir laden die standard_eval.csv aus allen vier Szenarien und extrahieren Servicerate, durchschnittliche/Median/90%-Wartezeit und Fahrzeugauslastung.

In [3]:
# Ergebnisordner und alle Szenarien
base_path = '../results'

# Definiere Altersgruppen
age_groups = {
    'Ältere (65+)': ['baseline', 'phone_support', 'simple_ui', 'full_support'],
    'Schüler (15-20)': ['students_15_20'],
    'Studenten (21-30)': ['students_21_30']
}

kennzahlen = []
for group_name, szenarien in age_groups.items():
    for sz in szenarien:
        eval_path = os.path.join(base_path, sz, 'standard_eval.csv')
        if not os.path.exists(eval_path):
            print(f"Fehlt: {eval_path}")
            continue
        df = pd.read_csv(eval_path, index_col=0)

        # Extract key metrics (using actual column name 'MoD_0')
        servicerate = df.loc['served online users [%]', 'MoD_0']
        avg_wait = df.loc['waiting time', 'MoD_0']
        med_wait = df.loc['waiting time (median)', 'MoD_0']
        p90_wait = df.loc['waiting time (90% quantile)', 'MoD_0']
        utilization = df.loc['% fleet utilization', 'MoD_0']

        kennzahlen.append({
            'Altersgruppe': group_name,
            'Szenario': sz,
            'Servicerate [%]': servicerate,
            'Ø Wartezeit [s]': avg_wait,
            'Median Wartezeit [s]': med_wait,
            '90%-Quantil Wartezeit [s]': p90_wait,
            'Fahrzeugauslastung [%]': utilization
        })

result_df = pd.DataFrame(kennzahlen)
result_df

,Altersgruppe,Szenario,Servicerate [%],Ø Wartezeit [s],Median Wartezeit [s],90%-Quantil Wartezeit [s],Fahrzeugauslastung [%]
0,Ältere (65+),baseline,0.857143,116.957260,120.000000,184.697423,2.648362
1,Ältere (65+),phone_support,0.857143,116.957260,120.000000,184.697423,2.634473
2,Ältere (65+),simple_ui,1.428571,184.174356,200.871779,288.000000,4.203918
3,Ältere (65+),full_support,1.714286,233.478630,250.435889,375.000000,6.169354
4,Schüler (15-20),students_15_20,1.428571,190.174356,200.871779,300.000000,5.190187
5,Studenten (21-30),students_21_30,1.714286,283.254810,300.000000,434.328541,4.590892


### 📊 Hinweis zur Metrikberechnung

Die Servicerate wird aus `standard_eval.csv` geladen. Diese Metrik wird von FleetPy automatisch berechnet als **Anteil der Anfragen mit `pickup_time != NaN`**, identisch zur Methode in H1/H2-Analysen.

**Warum nicht manuell berechnen wie in H1/H2?**
- `standard_eval.csv` enthält bereits aggregierte Metriken für alle Szenarien
- Reduziert Code-Duplikation und potenzielle Fehlerquellen
- Identisches Berechnungsverfahren wie H1/H2 (pickup_time-basiert)
- Konsistente Auswertung über alle Use Cases

## 3. Visualisierung: Servicerate und Wartezeiten im Vergleich
Wir plotten die Servicerate und die Wartezeiten für alle Szenarien als Balkendiagramm.

In [4]:
fig = px.bar(result_df, x='Szenario', y='Servicerate [%]',
             color='Altersgruppe',
             title='Servicerate je Szenario und Altersgruppe',
             text='Servicerate [%]')
fig.update_traces(texttemplate='%{text:.1f}', textposition='outside')
fig.show()

fig2 = px.bar(result_df, x='Szenario', y=['Ø Wartezeit [s]', 'Median Wartezeit [s]', '90%-Quantil Wartezeit [s]'],
              color='Altersgruppe',
              title='Wartezeiten je Szenario und Altersgruppe (in Sekunden)',
              barmode='group')
fig2.show()

## 4. Zusammenfassung und Interpretation

Die Ergebnisse zeigen, wie verschiedene Unterstützungsmaßnahmen die Servicerate und Wartezeiten für ältere Nutzer beeinflussen. Besonders die Kombination aus telefonischer Unterstützung und einer vereinfachten Benutzeroberfläche ("Full Support") führt zu einer deutlichen Verbesserung der Servicequalität. Die Wartezeiten sinken und die Servicerate steigt im Vergleich zum Baseline-Szenario signifikant an.

**Empfehlung:**
- Für eine inklusive On-Demand-Mobilität sollten Betreiber gezielt Support-Maßnahmen für Nutzer mit niedriger digitaler Kompetenz implementieren.
- Die Simulationsergebnisse unterstützen die Einführung von telefonischer Buchung und einer vereinfachten App-Oberfläche als wirksame Maßnahmen.

---

### Weiterführende Schritte

- Die Ergebnisse können für die wissenschaftliche Berichterstattung und die Ableitung von Handlungsempfehlungen genutzt werden.
- Für eine detaillierte Analyse können weitere Metriken (z.B. Kosten, Fahrzeugauslastung) ergänzt werden.
- Die Methodik ist auf andere Nutzergruppen und Regionen übertragbar.